# 01 Data Inspection (Stage 2)

Goal: understand the structure of `cell2celltrain.csv` before any cleaning or modelling.

This notebook only **looks** at the data. It does not change it. The raw file in `data/raw/` is read-only.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cell2celltrain.csv"
ID_COLUMN = "CustomerID"
TARGET_COLUMN = "Churn"
SAMPLE_ROWS = 5
TOP_CATEGORIES = 5
BYTES_PER_MB = 1024**2

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {RAW_DATA_PATH}. "
        "Run this notebook from the notebooks/ folder and check the CSV is in data/raw/."
    )

## 2. File check

Size and line count of the raw file, so we can compare with what pandas loads.

In [ ]:
file_size_mb = RAW_DATA_PATH.stat().st_size / BYTES_PER_MB

with RAW_DATA_PATH.open(encoding="utf-8") as raw_file:
    line_count = sum(1 for _ in raw_file)

print(f"File size: {file_size_mb:.1f} MB")
print(f"Lines in file (including header): {line_count:,}")

## 3. Load the data

If rows loaded is not equal to lines minus the header, some rows are broken or split across lines.

In [ ]:
df = pd.read_csv(RAW_DATA_PATH, encoding="utf-8")

expected_rows = line_count - 1
print(f"Shape (rows, columns): {df.shape}")
print(f"Rows expected from line count: {expected_rows:,}")
print(f"Rows loaded match line count: {len(df) == expected_rows}")

## 4. Column summary

Type, number of distinct values and missing values for every column.

In [ ]:
print("Columns per type:")
print(df.dtypes.astype(str).value_counts().to_string())

column_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean().mul(100).round(2),
    }
)
column_summary

## 5. Sample rows

`CustomerID` is left out on purpose: we do not print identifiers.

In [ ]:
df.drop(columns=ID_COLUMN).head(SAMPLE_ROWS)

## 6. Target: `Churn`

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts(dropna=False)
target_summary = pd.DataFrame(
    {"count": target_counts, "pct": target_counts.div(len(df)).mul(100).round(2)}
)
target_summary

## 7. Duplicates

- Identical rows (all columns the same)
- Identical rows apart from `CustomerID` (same customer data under two IDs?)
- Repeated `CustomerID` values (is each row really one customer?)

In [ ]:
full_row_duplicates = df.duplicated().sum()
duplicates_ignoring_id = df.drop(columns=ID_COLUMN).duplicated().sum()
repeated_ids = df[ID_COLUMN].duplicated().sum()

print(f"Identical rows: {full_row_duplicates:,}")
print(f"Identical rows ignoring {ID_COLUMN}: {duplicates_ignoring_id:,}")
print(f"Repeated {ID_COLUMN} values: {repeated_ids:,}")

## 8. Text columns

Number of categories and the most common values, to spot odd labels (for example `Unknown` or typos).

In [ ]:
text_columns = [
    column
    for column in df.columns
    if column != ID_COLUMN and not pd.api.types.is_numeric_dtype(df[column])
]

print(f"Text columns: {len(text_columns)}\n")
for column in text_columns:
    top_values = df[column].value_counts(dropna=False).head(TOP_CATEGORIES).to_dict()
    print(f"{column} ({df[column].nunique()} categories): {top_values}")

## 9. Number ranges

Look for impossible values, for example negative minutes or charges.

In [ ]:
df.drop(columns=ID_COLUMN).describe().T